# File 2 - CodeBERT Hyperparameter Tuning

Runs a mini grid search on a 5,000 sample subset to find the best CodeBERT
configuration for File 3.




## 0. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate

## 1. Imports & Setup

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


## 2. Configuration

In [ ]:
CODEBERT_MODEL  = "microsoft/codebert-base"
SUBSET_SIZE     = 5000     # small subset for fast tuning
TUNING_BATCH    = 32

# Search space
POOLING_OPTIONS = ["cls", "mean", "max"]
MAXLEN_OPTIONS  = [128, 256, 512]

#saving the result
OUTPUT_JSON = "best_hyperparams.json"


## 3. Load & Prepare Data

In [ ]:
def extract_code_from_prompt(text):
    if "The extract:" not in text:
        return ""
    code = text.split("The extract:")[-1]
    for marker in ("After examining the extract:", "Educational score:"):
        if marker in code:
            code = code.split(marker)[0]
    return code.strip()


In [ ]:
print(f"Loading {SUBSET_SIZE} samples from HuggingFace...")
ds = load_dataset("HuggingFaceTB/python-edu-annotations", split="train")
df = ds.to_pandas().sample(SUBSET_SIZE, random_state=42)

if "score" not in df.columns and "int_score" in df.columns:
    df = df.rename(columns={"int_score": "score"})

df["code"]  = df["prompt"].apply(extract_code_from_prompt)
df          = df.dropna(subset=["score", "code"])
df["score"] = df["score"].astype(int)
df          = df[df["score"].between(1, 5) & (df["code"].str.len() >= 20)]
df          = df.reset_index(drop=True)

print(f"Clean subset: {df.shape}")
print(df["score"].value_counts().sort_index().rename("count").to_frame())


## 4. Embedding Utilities

In [ ]:
class SimpleCodeDataset(Dataset):
    def __init__(self, codes, tokenizer, max_len):
        self.codes     = codes
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        return self.tokenizer(
            str(self.codes[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

@torch.no_grad()
def get_embeddings(codes, model, tokenizer, max_len, pooling):
    """
    Extract embeddings using the specified pooling strategy.
      cls  — take the [CLS] token (index 0)
      mean — average all token vectors
      max  — element-wise max across all token vectors
    """
    model.eval()
    ds     = SimpleCodeDataset(codes, tokenizer, max_len)
    loader = DataLoader(ds, batch_size=TUNING_BATCH, shuffle=False)
    all_embs = []

    for batch in loader:
        ids  = batch["input_ids"].squeeze(1).to(DEVICE)
        mask = batch["attention_mask"].squeeze(1).to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)

        if pooling == "cls":
            vecs = out.last_hidden_state[:, 0, :]
        elif pooling == "mean":
            vecs = out.last_hidden_state.mean(dim=1)
        elif pooling == "max":
            vecs = out.last_hidden_state.max(dim=1)[0]

        all_embs.append(vecs.cpu().numpy())

    return np.vstack(all_embs)


## 5. Load CodeBERT

In [ ]:
print(f"Loading '{CODEBERT_MODEL}'...")
tokenizer = AutoTokenizer.from_pretrained(CODEBERT_MODEL)
encoder   = AutoModel.from_pretrained(CODEBERT_MODEL).to(DEVICE)
encoder.eval()
print(f"Loaded — {sum(p.numel() for p in encoder.parameters()):,} params")


## 6. Grid Search

Runs 9 combinations (3 max_len × 3 pooling).  
Prints results row-by-row so you can monitor progress and stop early if needed.


In [ ]:
results   = []
best_f1   = 0.0
best_config = {}

print(f"{'MaxLen':<8} | {'Pooling':<8} | {'Macro-F1':<10} | {'Status'}")
print("-" * 46)

for m_len in MAXLEN_OPTIONS:
    for pool in POOLING_OPTIONS:

        # 1. Extract embeddings
        X = get_embeddings(df["code"].tolist(), encoder, tokenizer, m_len, pool)
        y = df["score"].values.astype(int)

        # 2. Quick train/val split — stratified
        X_tr, X_va, y_tr, y_va = train_test_split(
            X, y, test_size=0.20, stratify=y, random_state=42
        )

        # 3. Logistic Regression — fast, unbiased probe
        clf = LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        ).fit(X_tr, y_tr)

        f1 = f1_score(y_va, clf.predict(X_va), average="macro", zero_division=0)

        is_best = f1 > best_f1
        status  = "<-- BEST" if is_best else ""
        print(f"{m_len:<8} | {pool:<8} | {f1:.4f}     | {status}")

        results.append({"max_len": m_len, "pooling": pool, "macro_f1": float(f1)})

        if is_best:
            best_f1 = f1
            best_config = {
                "best_max_len": int(m_len),
                "best_pooling": pool,
                "f1":           float(f1),
            }

print("\n" + "=" * 46)
print(f"  WINNER : max_len={best_config['best_max_len']},  pooling={best_config['best_pooling']}")
print(f"  BEST MACRO-F1 : {best_config['f1']:.4f}")
print("=" * 46)


## 7. Full Results Table

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_res = pd.DataFrame(results)

# Pivot for heatmap
pivot = df_res.pivot(index="max_len", columns="pooling", values="macro_f1")
pivot = pivot[["cls", "mean", "max"]]   # fix column order

print(pivot.to_string())


In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlOrRd",
            linewidths=.5, ax=ax, cbar_kws={"shrink": .8},
            annot_kws={"size": 12})
ax.set_title("Grid Search — Macro-F1 (5,000 samples, LR probe)",
             fontweight="bold", fontsize=12)
ax.set_xlabel("Pooling Strategy")
ax.set_ylabel("Max Token Length")
plt.tight_layout()
plt.savefig("grid_search_heatmap.png", bbox_inches="tight", dpi=130)
plt.show()
print("Saved -> grid_search_heatmap.png")


In [ ]:
# Bar chart per pooling strategy
PALETTE = ["#2E4057", "#048A81", "#54C6EB"]
fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(MAXLEN_OPTIONS))
w = 0.25
for i, pool in enumerate(["cls", "mean", "max"]):
    vals = [df_res[(df_res["max_len"]==m) & (df_res["pooling"]==pool)]["macro_f1"].values[0]
            for m in MAXLEN_OPTIONS]
    bars = ax.bar([xi + i*w for xi in x], vals, w, label=pool,
                  color=PALETTE[i], edgecolor="white")
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
ax.set_xticks([xi + w for xi in x])
ax.set_xticklabels(MAXLEN_OPTIONS)
ax.set_xlabel("Max Token Length"); ax.set_ylabel("Macro-F1 (Val)")
ax.set_title("Grid Search — Macro-F1 by MaxLen & Pooling", fontweight="bold")
ax.legend(title="Pooling")
ax.set_ylim(0, min(1.0, df_res["macro_f1"].max() * 1.2))
plt.tight_layout()
plt.savefig("grid_search_bar.png", bbox_inches="tight", dpi=130)
plt.show()
print("Saved -> grid_search_bar.png")


## 8. Save Best Config to JSON

In [ ]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(best_config, f, indent=2)

print(f"Saved best config to: {OUTPUT_JSON}")
print(json.dumps(best_config, indent=2))
